# 🎯 Step 3: Model Evaluation & Visual Inference
Testing the trained model on unseen data to verify real-world performance.

In [1]:
import os
import cv2
import time
import random
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image, display

# 1. Setup Directories
BASE_DIR = Path.cwd().resolve()
MODEL_PATH = BASE_DIR / 'saved_models' / 'best.pt'
DATA_YAML = BASE_DIR / 'HelmetDataset' / 'data.yaml'
TEST_IMG_DIR = BASE_DIR / 'HelmetDataset' / 'test' / 'images'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

# 2. Load Model
model = YOLO(str(MODEL_PATH))
print(f"✅ Model successfully loaded from: {MODEL_PATH}")


✅ Model loaded from C:\My Space\GitHub Repo\Traffic-Violation-Detection-Deep-Learning\saved_models\best.pt


In [2]:
# =====================================================================
# 1. COMPREHENSIVE MODEL EVALUATION & ALL METRICS
# =====================================================================
print("🔍 Running evaluation on unseen test dataset...")
metrics = model.val(data=str(DATA_YAML), split="test", verbose=False)

# Extract Core Detection Metrics
p = float(metrics.box.mp)
r = float(metrics.box.mr)
map50 = float(metrics.box.map50)
map95 = float(metrics.box.map)
f1 = 2 * (p * r) / (p + r + 1e-6)

# Extract Speed & Latency (FPS)
speed = metrics.speed
prep_ms = speed.get('preprocess', 0.0)
inf_ms = speed.get('inference', 0.0)
post_ms = speed.get('postprocess', 0.0)
total_latency_ms = prep_ms + inf_ms + post_ms
fps = 1000.0 / total_latency_ms if total_latency_ms > 0 else 0.0

# 1. Overall Accuracy Table
print("=" * 72)
print("🎯 1. OVERALL OBJECT DETECTION ACCURACY")
print("=" * 72)
print(f"  • Detection Accuracy (mAP@50)    : {map50 * 100:>6.2f}%  (IoU >= 0.50)")
print(f"  • Strict Accuracy (mAP@50-95)    : {map95 * 100:>6.2f}%  (Averaged IoU 0.50-0.95)")
print(f"  • Precision (Correct Detections) : {p * 100:>6.2f}%  (TP / (TP + FP))")
print(f"  • Recall (Captured Ground-Truth) : {r * 100:>6.2f}%  (TP / (TP + FN))")
print(f"  • Balanced Accuracy (F1-Score)   : {f1 * 100:>6.2f}%  (Harmonic mean of P & R)")
print("=" * 72)

# 2. Speed and Hardware Telemetry
print("\n⚡ 2. INFERENCE SPEED & LATENCY (FPS)")
print("-" * 72)
print(f"  • Pre-process Latency  : {prep_ms:.2f} ms")
print(f"  • Model Inference Time : {inf_ms:.2f} ms")
print(f"  • Post-process Latency : {post_ms:.2f} ms")
print(f"  • Total Frame Latency  : {total_latency_ms:.2f} ms per image")
print(f"  • Processing Speed     : ~{fps:.1f} FPS (Frames Per Second)")
print("-" * 72)

# 3. Per-Class Accuracy Breakdown
print("\n📋 3. PER-CLASS ACCURACY BREAKDOWN")
print("-" * 72)
header = f"{'Class ID':<10} {'Class Name':<20} {'Precision':<12} {'Recall':<12} {'F1-Score':<12} {'mAP@50':<12}"
print(header)
print("-" * 72)
for i, name in model.names.items():
    cp = float(metrics.box.p[i]) if i < len(metrics.box.p) else 0.0
    cr = float(metrics.box.r[i]) if i < len(metrics.box.r) else 0.0
    cf1 = 2 * (cp * cr) / (cp + cr + 1e-6)
    cmap50 = float(metrics.box.all_ap[i, 0]) if hasattr(metrics.box, 'all_ap') else float(metrics.box.maps[i])
    print(f"{i:<10} {name:<20} {cp * 100:>8.2f}%    {cr * 100:>8.2f}%    {cf1 * 100:>8.2f}%    {cmap50 * 100:>8.2f}%")
print("=" * 72)

# 4. Display Evaluation Curve Charts
charts_dir = Path(metrics.save_dir) if hasattr(metrics, 'save_dir') and Path(metrics.save_dir).exists() else (BASE_DIR / 'runs' / 'detect' / 'val')
chart_files = [
    charts_dir / 'confusion_matrix.png',
    charts_dir / 'BoxPR_curve.png',
    charts_dir / 'BoxF1_curve.png'
]

print(f"\n📁 Evaluation Charts Loaded From: {charts_dir}\n")
for chart_path in chart_files:
    if chart_path.exists():
        img = cv2.imread(str(chart_path))
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(9, 6))
            plt.imshow(img)
            plt.title(chart_path.name, fontsize=13, fontweight='bold')
            plt.axis('off')
            plt.show()
    else:
        print(f"ℹ️ Chart not found: {chart_path.name}")


                   all         76        136      0.913      0.901      0.949       0.65
           With Helmet         61        102      0.892      0.971      0.975      0.709
        Without Helmet         19         34      0.934      0.832      0.923       0.59
Speed: 1.7ms preprocess, 44.6ms inference, 0.0ms loss, 0.4ms postprocess per image
📊 OVERALL TEST ACCURACY AND DETECTION METRICS
🎯 Overall Detection Accuracy (mAP@50) : 94.90%
🎯 Precision (Correct Detections)       : 91.28%
🎯 Recall (Captured Ground-Truths)     : 90.14%
🎯 F1-Score (Balanced Accuracy)        : 90.71%
🎯 Strict Accuracy (mAP@50-95)         : 64.98%

📋 PER-CLASS ACCURACY BREAKDOWN:
Class                Precision    Recall       F1-Score     mAP@50      
--------------------------------------------------------------------
With Helmet             89.17%       97.06%       92.94%       97.48%
Without Helmet          93.40%       83.23%       88.02%       92.32%


In [3]:
# =====================================================================
# 2. VISUAL INFERENCE ON RANDOM TEST SAMPLES
# =====================================================================
images = [f for f in os.listdir(TEST_IMG_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
random.seed(42)
samples = random.sample(images, min(9, len(images)))

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, img_name in enumerate(samples):
    res = model.predict(str(TEST_IMG_DIR / img_name), conf=0.3, verbose=False)[0]
    annotated = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)
    axes[i].imshow(annotated)
    axes[i].set_title(img_name, fontsize=10)
    axes[i].axis('off')

plt.tight_layout()
plt.show()


🖼️ Visual Inference on 9 Random Test Images:
   - BikesHelmets210.png    -> 1 object(s): ['With Helmet']
   - BikesHelmets119.png    -> 1 object(s): ['Without Helmet']
   - BikesHelmets342.png    -> 2 object(s): ['With Helmet', 'With Helmet']
   - BikesHelmets317.png    -> 1 object(s): ['With Helmet']
   - BikesHelmets309.png    -> 1 object(s): ['With Helmet']
   - BikesHelmets23.png     -> 3 object(s): ['With Helmet', 'With Helmet', 'With Helmet']
   - BikesHelmets209.png    -> 2 object(s): ['With Helmet', 'With Helmet']
   - BikesHelmets726.png    -> 1 object(s): ['With Helmet']
   - BikesHelmets185.png    -> 1 object(s): ['With Helmet']
